In [11]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("dataset/machine_failure.csv")

df = df.drop(columns=["UDI", "Product ID", "Type"])

feature_cols = [col for col in df.columns if col not in 
                ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]]

X = df[feature_cols]

targets = ["TWF", "HDF", "PWF", "OSF", "RNF"]

# =========================
# SPLIT
# =========================
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 🔥 STAGE 1: LOCAL OUTLIER FACTOR (IMPROVED ANOMALY DETECTION)
# =========================
lof = LocalOutlierFactor(
    n_neighbors=35,
    contamination=0.05,
    novelty=True
)

lof.fit(X_train_scaled)

def get_anomaly_score(model, X):
    raw = model.decision_function(X)

    # convert to 0–1 using sigmoid (stable + meaningful)
    return 1 / (1 + np.exp(-raw))

# =========================
# 🔥 STAGE 2: FAILURE MODELS (SMOTE + RF)
# =========================
models = {}

for target in targets:
    print(f"\nTraining {target}")

    y_train = df.loc[X_train.index, target]
    y_test = df.loc[X_test.index, target]

    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=14,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_res, y_res)
    models[target] = model

    y_pred = model.predict(X_test_scaled)

    print(classification_report(y_test, y_pred, zero_division=0))

# =========================
# 🔥 STAGE 3: FINAL DECISION LOGIC
# =========================
def final_decision(anomaly_score, failure_vector):
    if anomaly_score > 0.7:
        return "FAILURE"
    elif anomaly_score > 0.4 or sum(failure_vector) > 0:
        return "WARNING"
    else:
        return "NORMAL"

# =========================
# PREDICTION PIPELINE
# =========================
def predict(X_input):
    X_scaled = scaler.transform(X_input)

    anomaly_score = get_anomaly_score(lof, X_scaled)[0]

    failure_vector = [
        int(models["TWF"].predict(X_scaled)[0]),
        int(models["HDF"].predict(X_scaled)[0]),
        int(models["PWF"].predict(X_scaled)[0]),
        int(models["OSF"].predict(X_scaled)[0]),
        int(models["RNF"].predict(X_scaled)[0]),
    ]

    return {
        "anomaly_score": float(anomaly_score),
        "failure_vector": failure_vector,
        "decision": final_decision(anomaly_score, failure_vector)
    }

# =========================
# TEST SAMPLE
# =========================
sample = pd.DataFrame([{
    "Air temperature [K]": 298.2,
    "Process temperature [K]": 308.5,
    "Rotational speed [rpm]": 1400,
    "Torque [Nm]": 65.0,
    "Tool wear [min]": 191
}])
sampleFaultless = pd.DataFrame([{
    "Air temperature [K]": 298.2,
    "Process temperature [K]": 308.5,
    "Rotational speed [rpm]": 1400,
    "Torque [Nm]": 30,
    "Tool wear [min]": 106
}])

print("\nFINAL OUTPUT:")
print(predict(sample))
print("\nFAULTLESS SAMPLE OUTPUT:")
print(predict(sampleFaultless))


# =========================
# SAVE MODELS
# =========================
joblib.dump(lof, "lof_model.joblib")
joblib.dump(models, "rf_models.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(feature_cols, "feature_cols.joblib")

print("\nModels saved successfully!")


Training TWF
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      1989
           1       0.09      0.55      0.15        11

    accuracy                           0.97      2000
   macro avg       0.54      0.76      0.57      2000
weighted avg       0.99      0.97      0.98      2000


Training HDF
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1983
           1       0.72      0.76      0.74        17

    accuracy                           1.00      2000
   macro avg       0.86      0.88      0.87      2000
weighted avg       1.00      1.00      1.00      2000


Training PWF
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1980
           1       0.59      0.95      0.73        20

    accuracy                           0.99      2000
   macro avg       0.80      0.97      0.86      2000
weighted avg       1.00      0.9